In [3]:
import argparse, json, os, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO


In [4]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
COCO_DIR    = Path("./coco")
VAL_IMG_DIR = COCO_DIR / "val2017" 
ANN_FILE    = COCO_DIR / "annotations/instances_val2017.json"
IMG_SIZE    = 640
DEVICE      = "cuda"      # "cuda" | "cpu" | "mps"
CONF        = 0.001       # low → recall-friendly (standard for mAP eval)
IOU         = 0.65
WARMUP      = 3
RESULTS_CSV = "results_yolov11.csv"

# Note: Ultralytics uses "yolo11" (no 'v') for v11 weight names
MODELS = [
    "yolo11n.pt",
    "yolo11s.pt",
    "yolo11m.pt",
    "yolo11l.pt",
    "yolo11x.pt",
]

In [5]:
# ── COCO DOWNLOAD ─────────────────────────────────────────────────────────────
def download_coco():
    COCO_DIR.mkdir(parents=True, exist_ok=True)
    files = {
        "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
        "annotations_trainval2017.zip":
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    }
    for fname, url in files.items():
        dest = COCO_DIR / fname
        if not dest.exists():
            print(f"Downloading {fname} ...")
            urllib.request.urlretrieve(url, dest)
            with zipfile.ZipFile(dest) as z:
                z.extractall(COCO_DIR)
            print(f"  ✔ Extracted {fname}")

In [6]:
# ── COCO CATEGORY MAP ─────────────────────────────────────────────────────────
def build_id_map(ann_file):
    """Map 0-indexed YOLO class → COCO category_id."""
    with open(ann_file) as f:
        data = json.load(f)
    cats = sorted(data["categories"], key=lambda c: c["id"])
    return {i: c["id"] for i, c in enumerate(cats)}

In [7]:
# ── EVALUATION ────────────────────────────────────────────────────────────────
def coco_eval(coco_gt, predictions):
    if not predictions:
        return 0.0, 0.0
    coco_dt = coco_gt.loadRes(predictions)
    ev = COCOeval(coco_gt, coco_dt, "bbox")
    ev.evaluate()
    ev.accumulate()
    ev.summarize()
    return round(ev.stats[1], 4), round(ev.stats[0], 4)

In [8]:
# ── SINGLE MODEL BENCHMARK ────────────────────────────────────────────────────
def benchmark(model_name, img_ids, coco_gt, id_map):
    print(f"\n{'='*55}")
    print(f"  Model : {model_name}")
    print(f"{'='*55}")

    model = YOLO(model_name)

    weight_path = Path(model.ckpt_path) if hasattr(model, "ckpt_path") else Path(model_name)
    size_mb  = weight_path.stat().st_size / 1e6 if weight_path.exists() else 0.0
    params_m = sum(p.numel() for p in model.model.parameters()) / 1e6

    # Warm-up
    dummy = str(next(VAL_IMG_DIR.glob("*.jpg")))
    for _ in range(WARMUP):
        model(dummy, imgsz=IMG_SIZE, device=DEVICE,
              conf=CONF, iou=IOU, verbose=False)

    predictions, latencies = [], []

    for img_id in tqdm(img_ids, desc=model_name, unit="img"):
        img_path = VAL_IMG_DIR / f"{img_id:012d}.jpg"
        if not img_path.exists():
            continue

        t0 = time.perf_counter()
        results = model(str(img_path), imgsz=IMG_SIZE, device=DEVICE,
                        conf=CONF, iou=IOU, verbose=False)[0]
        latencies.append((time.perf_counter() - t0) * 1000)

        boxes  = results.boxes.xyxy.cpu().numpy()
        scores = results.boxes.conf.cpu().numpy()
        cls    = results.boxes.cls.cpu().numpy().astype(int)

        for box, score, c in zip(boxes, scores, cls):
            x1, y1, x2, y2 = box
            predictions.append({
                "image_id":    img_id,
                "category_id": id_map.get(c, c + 1),
                "bbox":        [float(x1), float(y1),
                                float(x2 - x1), float(y2 - y1)],
                "score":       float(score),
            })

    map50, map5095 = coco_eval(coco_gt, predictions)
    avg_ms = round(float(np.mean(latencies)), 2)

    row = {
        "model":         model_name.replace(".pt", ""),
        "mAP@0.5":       map50,
        "mAP@0.5:0.95":  map5095,
        "speed_ms/img":  avg_ms,
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }
    print(f"\n  ▶ mAP@0.5        : {map50}")
    print(f"  ▶ mAP@0.5:0.95   : {map5095}")
    print(f"  ▶ Speed (ms/img)  : {avg_ms}")
    print(f"  ▶ Size (MB)       : {row['size_MB']}")
    print(f"  ▶ Params (M)      : {row['params_M']}")
    del model
    return row


In [11]:
import argparse
import traceback
import gc
import torch
import pandas as pd
from pycocotools.coco import COCO

# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--quick", action="store_true",
                        help="Run on 500 images instead of full 5k")
    
    # Use parse_known_args to ignore Jupyter's extra args
    args, unknown = parser.parse_known_args()

    download_coco()

    coco_gt = COCO(str(ANN_FILE))
    img_ids = coco_gt.getImgIds()
    if args.quick:
        img_ids = img_ids[:500]
        print(f"[Quick mode] Using {len(img_ids)} images.")

    id_map = build_id_map(ANN_FILE)
    rows = []

    for model_name in MODELS:
        try:
            row = benchmark(model_name, img_ids, coco_gt, id_map)
            rows.append(row)
        except Exception as e:
            print(f"  ⚠ Skipped {model_name}: {e}")
            traceback.print_exc()
        finally:
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            print(f"  🧹 VRAM freed — "
                  f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | "
                  f"Cached: {torch.cuda.memory_reserved()/1e9:.2f} GB")

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)

    print(f"\n{'='*55}")
    print("  YOLOv11 — Final Results")
    print(f"{'='*55}")
    print(df.to_string(index=False))
    print(f"\n✅ Saved → {RESULTS_CSV}")

if __name__ == "__main__":
    main()

loading annotations into memory...
Done (t=0.66s)
creating index...
index created!

  Model : yolo11n.pt


yolo11n.pt: 100%|██████████| 5000/5000 [01:40<00:00, 49.94img/s]


Loading and preparing results...
DONE (t=1.23s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=33.55s).
Accumulating evaluation results...
DONE (t=5.85s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.387
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.545
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.418
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.194
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.424
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.561
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.310
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.506
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.553
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

yolo11s.pt: 100%|██████████| 5000/5000 [01:40<00:00, 49.76img/s]


Loading and preparing results...
DONE (t=0.92s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=30.87s).
Accumulating evaluation results...
DONE (t=4.86s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.456
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.626
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.490
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.278
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.501
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.625
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.348
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.566
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.611
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

...

Loading and preparing results...
DONE (t=0.96s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.87s).
Accumulating evaluation results...
DONE (t=4.33s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.505
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.675
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.547
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.334
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.560
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.663
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.373
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.612
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.656
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

yolo11l.pt: 100%|██████████| 5000/5000 [05:48<00:00, 14.33img/s]


Loading and preparing results...
DONE (t=2.44s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=68.71s).
Accumulating evaluation results...
DONE (t=17.21s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.522
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.689
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.566
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.353
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.578
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.683
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.382
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.627
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.669
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

yolo11x.pt: 100%|██████████| 5000/5000 [06:13<00:00, 13.38img/s]


Loading and preparing results...
DONE (t=0.80s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=24.74s).
Accumulating evaluation results...
DONE (t=3.77s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.535
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.705
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.580
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.371
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.586
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.690
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.389
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.638
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.680
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

In [12]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(RESULTS_CSV)

# Style the table
styled = (
    df.style
    .set_caption("YOLOv11 — Benchmark Results on COCO val2017")
    .format({
        "mAP@0.5":      "{:.4f}",
        "mAP@0.5:0.95": "{:.4f}",
        "speed_ms/img": "{:.1f} ms",
        "size_MB":      "{:.1f} MB",
        "params_M":     "{:.1f} M",
    })
    .highlight_max(subset=["mAP@0.5", "mAP@0.5:0.95"], color="black")
    .highlight_min(subset=["speed_ms/img", "size_MB", "params_M"], color="black")
    .set_properties(**{"text-align": "center", "font-size": "13px"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "15px"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "th",
         "props": [("background-color", "#000000"), ("color", "white"),
                   ("font-size", "13px"), ("text-align", "center"), ("padding", "8px")]},
        {"selector": "tr:hover",
         "props": [("background-color", "#0B0B0B")]},
    ])
    .hide(axis="index")
)

display(styled)

model,mAP@0.5,mAP@0.5:0.95,speed_ms/img,size_MB,params_M
yolo11n,0.5453,0.3866,18.9 ms,5.6 MB,2.6 M
yolo11s,0.6257,0.4561,19.1 ms,19.3 MB,9.5 M
yolo11m,0.6748,0.5052,31.5 ms,40.7 MB,20.1 M
yolo11l,0.6888,0.5217,67.4 ms,51.4 MB,25.4 M
yolo11x,0.7047,0.5351,73.3 ms,114.6 MB,57.0 M
